← [Overview](00_overview.ipynb)

# Representation strategies

Clustering decides *which* periods group together; **representation** decides what each
cluster's single profile actually looks like. In tsam these are **separate steps**: the
`representation=` lever is independent of the clustering method (see
[Partitional clustering](02_partitional_clustering.ipynb) for that distinction). Every
method just sets a sensible default representative, which you can override freely.

This notebook covers the five strategies — `mean`, `medoid`, `maxoid`, `Distribution`,
`MinMaxMean` — and how they trade reconstruction accuracy against keeping *real* profiles or
the value *distribution*. The follow-on [Rescaling](07_rescaling.ipynb) notebook then
corrects the totals and returns the profiles to physical units.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig, SegmentConfig
from tsam.config import Distribution, MinMaxMean

pio.renderers.default = "notebook_connected"

# Real 6-week dataset — representation differences show clearest with many periods.
raw = pd.read_csv("../testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]
print("real:", data.shape)

real: (1008, 4)


---

## The five strategies

| Strategy | What it picks | Use case |
|---|---|---|
| `mean` | Centroid (average) | Minimises within-cluster variance |
| `medoid` | Real period closest to centroid | Physically realistic profiles |
| `maxoid` | Real period most dissimilar to others | Spread/diversity |
| `Distribution` | Re-sorted values matching duration curve | Preserves value distribution |
| `MinMaxMean` | Per-column mix of min/max/mean | Fine-grained control |

Default representations by method:
- `kmeans`, `averaging` → `mean`
- `kmedoids`, `hierarchical`, `contiguous` → `medoid`
- `kmaxoids` → `maxoid`

**TSAM configuration for representation strategies:**

In [2]:
# The representation= parameter is set on ClusterConfig (or SegmentConfig).
# String shortcuts:
cfg_mean    = ClusterConfig(method="hierarchical", representation="mean")
cfg_medoid  = ClusterConfig(method="hierarchical", representation="medoid")
cfg_maxoid  = ClusterConfig(method="kmaxoids",    representation="maxoid")

# Typed objects (additional options):
cfg_dist_cluster = ClusterConfig(
    method="hierarchical",
    representation=Distribution(scope="cluster"),   # per-cluster duration curve
)
cfg_dist_global = ClusterConfig(
    method="hierarchical",
    representation=Distribution(scope="global"),    # overall duration curve
)
cfg_minmaxmean = ClusterConfig(
    method="hierarchical",
    representation=MinMaxMean(max_columns=["Load"], min_columns=[]),
)

# SegmentConfig also accepts representation=:
cfg_seg_medoid = SegmentConfig(n_segments=6, representation="medoid")

print('mean:        ', cfg_mean)
print('medoid:      ', cfg_medoid)
print('dist_cluster:', cfg_dist_cluster)
print('minmaxmean:  ', cfg_minmaxmean)
print('seg_medoid:  ', cfg_seg_medoid)

mean:         ClusterConfig(include_period_sums=False, method='hierarchical', representation='mean', scale_by_column_means=False, solver='highs', use_duration_curves=False)
medoid:       ClusterConfig(include_period_sums=False, method='hierarchical', representation='medoid', scale_by_column_means=False, solver='highs', use_duration_curves=False)
dist_cluster: ClusterConfig(include_period_sums=False, method='hierarchical', representation=Distribution(scope='cluster', preserve_minmax=False), scale_by_column_means=False, solver='highs', use_duration_curves=False)
minmaxmean:   ClusterConfig(include_period_sums=False, method='hierarchical', representation=MinMaxMean(max_columns=['Load'], min_columns=[]), scale_by_column_means=False, solver='highs', use_duration_curves=False)
seg_medoid:   SegmentConfig(n_segments=6, representation='medoid')


In [3]:
# Compare representations on the real dataset
reps = {
    "mean": "mean",
    "medoid": "medoid",
    "maxoid": "maxoid",
    "distribution": Distribution(scope="cluster"),
    "distribution_global": Distribution(scope="global"),
    "minmax_mean": MinMaxMean(max_columns=["Load"], min_columns=[]),
}
results_rep = {}
for name, rep in reps.items():
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=rep),
    )
    results_rep[name] = r

summary_rep = pd.DataFrame({
    name: {"weighted_rmse": round(r.accuracy.weighted_rmse, 4)}
    for name, r in results_rep.items()
}).T
print("Representation comparison (hierarchical k=6, real dataset):")
summary_rep

Representation comparison (hierarchical k=6, real dataset):


,weighted_rmse
mean,0.1022
medoid,0.1235
maxoid,0.1506
distribution,0.1362
distribution_global,0.1154
minmax_mean,0.1084


### Distribution representation — keeping the duration curve

The `mean` representative flattens the peaks of each cluster, so the reconstructed
**duration curve** sits inside the original. The `Distribution` representative instead
re-sorts values to match the cluster's duration curve, trading temporal shape for a far
better value distribution.

In [4]:
r_mean = results_rep["mean"]
r_dist = results_rep["distribution"]

r_mean.plot.compare(
    columns=["Load"],
    mode="duration_curve",
    title="Mean representation — Load duration curve vs original",
)

In [5]:
r_dist.plot.compare(
    columns=["Load"],
    mode="duration_curve",
    title="Distribution representation — Load duration curve vs original",
)

---

**Next:**
* [Rescaling & denormalisation](07_rescaling.ipynb) — correct the totals and return to physical units
* [Extreme periods](08_extreme_periods.ipynb) — preserving peak values no representation can recover
* [Representations how-to](../how-to/representations.ipynb) — full worked examples for every strategy